# Feature Extraction — ASVspoof 2019 LA Dataset
Extracts rf_features (MFCC mean + std), 1dcnn_features (MFCC matrix), and hybrid_features (mel spectrogram).

**Uses CPU runtime only**

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q numpy pandas soundfile librosa tqdm

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import os
import gc

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm import tqdm
from google.colab import drive

In [ ]:
# ── Cell 3: Confirm CPU runtime ────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'WARNING: You are on a GPU ({torch.cuda.get_device_name(0)}). '
          'Switch to CPU runtime to save compute units!')
else:
    print('CONFIRMED: CPU-only runtime. Good to go.')

CONFIRMED: CPU-only runtime. Good to go.


In [ ]:
# ── Cell 4: Mount Drive ────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


---
## Step 1 — Paths

In [ ]:
# ── Cell 5: Paths ──────────────────────────────────────────────────────────────
BASE_PATH = '/content/drive/MyDrive/Datasets/ASVSpoof2019_Dataset/LA'

DEV_PATH   = (f'{BASE_PATH}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt',  'dev')
EVAL_PATH  = (f'{BASE_PATH}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt', 'eval')
TRAIN_PATH = (f'{BASE_PATH}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt','train')

TRAIN_OUT = '/content/drive/MyDrive/ASV_Final_Features_train_v2.pkl'
DEV_OUT   = '/content/drive/MyDrive/ASV_Final_Features_dev_v2.pkl'
EVAL_OUT  = '/content/drive/MyDrive/ASV_Final_Features_eval_v2.pkl'

print('Paths configured.')

Paths configured.


---
## Step 2 — Load Labels

In [ ]:
# ── Cell 6: Load labels from protocol txt files ────────────────────────────────
def load_labels(file_path):
    column_names = ['speaker_id', 'file_id', 'dash1', 'system_id', 'label']
    sample = pd.read_csv(file_path[0], sep=r'\s+', names=column_names,
                         usecols=['file_id', 'label'])
    label_lookup = dict(zip(sample['file_id'], sample['label']))
    print(f'Loaded {len(label_lookup)} labels from {file_path[1]} file')
    return label_lookup

train_labels = load_labels(TRAIN_PATH)
dev_labels   = load_labels(DEV_PATH)
eval_labels  = load_labels(EVAL_PATH)

Loaded 25380 labels from train file
Loaded 24844 labels from dev file
Loaded 71237 labels from eval file


---
## Step 3 — Feature Extraction

In [ ]:
# ── Cell 7: Feature extraction function ───────────────────────────────────────
def extract_all_features_from_samples(file_path, n_mfcc=40, n_mels=128):
    """
    Returns three feature representations:
      rf_features     : (80,)     MFCC mean + std concatenated
      1dcnn_features  : (T, 40)   full MFCC matrix transposed
      hybrid_features : (T, 128)  mel spectrogram in dB transposed
    """
    audio, sr = sf.read(file_path)

    # Fixed 4-second length — pad short, trim long
    target_len = sr * 4
    if len(audio) > target_len:
        audio = audio[:target_len]
    else:
        audio = np.pad(audio, (0, target_len - len(audio)), mode='constant')

    # ── MFCC ──────────────────────────────────────────────────────────────────
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)  # (40, T)

    # rf_features: mean + std across time → (80,)
    mfcc_mean   = np.mean(mfcc, axis=1)                # (40,)
    mfcc_std    = np.std(mfcc,  axis=1)                # (40,)
    rf_features = np.concatenate([mfcc_mean, mfcc_std])# (80,)

    # 1dcnn_features: full matrix → (T, 40)
    dcnn_features = mfcc.T

    # ── Mel spectrogram ───────────────────────────────────────────────────────
    mels            = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
    spec_db         = librosa.power_to_db(mels, ref=np.max)  # (128, T)
    hybrid_features = spec_db.T                               # (T, 128)

    return rf_features, dcnn_features, hybrid_features

In [ ]:
# ── Cell 8: Extraction pipeline ───────────────────────────────────────────────
def feature_extraction_sprint(sample_set, labels):
    """
    sample_set : (audio_dir, save_path, split_name)
    labels     : dict mapping file_id -> 'bonafide' or 'spoof'
    """
    processed_data = []

    for root, dirs, files in os.walk(sample_set[0]):
        for filename in tqdm(files, desc=f'Processing {sample_set[2]}'):
            if not filename.endswith('.flac'):
                continue

            file_id = filename.replace('.flac', '')
            if file_id not in labels:
                continue

            full_path = os.path.join(root, filename)
            try:
                f_rf, f_1dcnn, f_hybrid = extract_all_features_from_samples(full_path)
                processed_data.append({
                    'file_id'        : file_id,
                    'label'          : 0 if labels[file_id] == 'bonafide' else 1,
                    'rf_features'    : f_rf,
                    '1dcnn_features' : f_1dcnn,
                    'hybrid_features': f_hybrid
                })
            except Exception as e:
                print(f'  [!] Skipped {filename}: {e}')
                continue

    if processed_data:
        df = pd.DataFrame(processed_data)
        df.to_pickle(sample_set[1])
        print(f'\nSUCCESS: {sample_set[2].upper()} ({len(df)} samples) saved to {sample_set[1]}')
        del processed_data, df
        gc.collect()
    else:
        print(f'\n[!] No samples processed for {sample_set[2]} — check audio path.')

---
## Step 4 — Copy Audio to Local VM (for faster extraction in Colab)

In [ ]:
# ── Cell 9: Copy train audio to local VM ──────────────────────────────────────
#!mkdir -p /content/train_local
#!cp -r '{BASE_PATH}/ASVspoof2019_LA_train/flac/' /content/train_local/

Uncomment lines above to copy train audio to local VM.


In [ ]:
# ── Cell 10: Copy dev audio to local VM ───────────────────────────────────────
#!mkdir -p /content/dev_local
#!cp -r '{BASE_PATH}/ASVspoof2019_LA_dev/flac/' /content/dev_local/

Uncomment lines above to copy dev audio to local VM.


In [ ]:
# ── Cell 11: Copy eval audio to local VM ──────────────────────────────────────
# Eval set is large — unzip from Drive to local VM
ZIP_PATH = '/content/drive/MyDrive/Datasets/ASVspoof2019_LA_eval.zip'
!cp "{ZIP_PATH}" /content/eval.zip
!unzip -q /content/eval.zip -d /content/eval_local/
!rm -rf /content/eval_local/__MACOSX
!find /content/eval_local/ -name "._*" -delete

Uncomment lines above to copy eval audio to local VM.


---
## Step 5 — Run Extraction

In [ ]:
# ── Cell 12: Run extraction — Train ───────────────────────────────────────────
train_sample_set = ('/content/train_local/', TRAIN_OUT, 'train')
feature_extraction_sprint(train_sample_set, train_labels)

Processing train: 0it [00:00, ?it/s]
Processing train: 100%|██████████| 25380/25380 [12:45<00:00, 33.14it/s]



SUCCESS: TRAIN (25380 samples) saved to /content/drive/MyDrive/ASV_Final_Features_train_v2.pkl


In [ ]:
# ── Cell 13: Run extraction — Dev ─────────────────────────────────────────────
dev_sample_set = ('/content/dev_local/', DEV_OUT, 'dev')
feature_extraction_sprint(dev_sample_set, dev_labels)

Processing dev: 0it [00:00, ?it/s]
Processing dev: 100%|██████████| 24986/24986 [12:24<00:00, 33.57it/s]



SUCCESS: DEV (24844 samples) saved to /content/drive/MyDrive/ASV_Final_Features_dev_v2.pkl


In [ ]:
# ── Cell 14: Run extraction — Eval ────────────────────────────────────────────
eval_sample_set = ('/content/eval_local/ASVspoof2019_LA_eval/flac/', EVAL_OUT, 'eval')
feature_extraction_sprint(eval_sample_set, eval_labels)

Processing eval: 100%|██████████| 71933/71933 [37:12<00:00, 32.22it/s]



SUCCESS: EVAL (71237 samples) saved to /content/drive/MyDrive/ASV_Final_Features_eval_v2.pkl


---
## Step 6 — Verify Output

In [ ]:
# ── Cell 15: Verify output ────────────────────────────────────────────────────
for name, path in [('Train', TRAIN_OUT), ('Dev', DEV_OUT), ('Eval', EVAL_OUT)]:
    df = pd.read_pickle(path)
    print(f'\n── {name} ────────────────────────────────────────────')
    print(f'  Samples : {len(df)}')
    print(f'  Labels  : {df["label"].value_counts().to_dict()}')
    print(f'  rf_features     shape: {df["rf_features"].iloc[0].shape}')      # expect (80,)
    print(f'  1dcnn_features  shape: {df["1dcnn_features"].iloc[0].shape}')   # expect (T, 40)
    print(f'  hybrid_features shape: {df["hybrid_features"].iloc[0].shape}')  # expect (T, 128)
    del df
    gc.collect()


── Train ────────────────────────────────────────────
  Samples : 25380
  Labels  : {1: 22800, 0: 2580}
  rf_features     shape: (80,)
  1dcnn_features  shape: (126, 40)
  hybrid_features shape: (126, 128)

── Dev ────────────────────────────────────────────
  Samples : 24844
  Labels  : {1: 22296, 0: 2548}
  rf_features     shape: (80,)
  1dcnn_features  shape: (126, 40)
  hybrid_features shape: (126, 128)

── Eval ────────────────────────────────────────────
  Samples : 71237
  Labels  : {1: 63882, 0: 7355}
  rf_features     shape: (80,)
  1dcnn_features  shape: (126, 40)
  hybrid_features shape: (126, 128)
